# Pendulum example

- a simple pendulum (a mass on a string) can be used to measure the gravitational acceleration g
- such a measurement was performed by students in a lab class and the results were recorded and stored in a CSV (comma separated values) file
- the measurements are a mix of "first pendulum" (large uncertainty) and "best pendulum" (smaller uncertainty) measurements

- __First step: Let's look at the raw data__

In [ ]:
# open the text file containing data

f = open("../../Data/pendulumData.csv","r")

# Use a while loop to read the data line-by-line until the end of file
flag = True
i = 0
while flag:
    s = f.readline()
    if s:
        print(i,s)
        i = i + 1
    else:
        # reached end of file; stop while loop
        flag = False
        
# close the data file
f.close()


__Second step: Read in the raw data, make a histogram, plot the data and examine it__

In [ ]:
#### import numpy as np
import matplotlib.pylab as plt
import numpy as np

# open a text file containing data

f = open("../../Data/pendulumData.csv","r")

#read one line of data and discard header line

s = f.readline()

# define empty array to store data values
g = []

#the while loop will run until the end of file or i == nevent
flag = True
i = 0
while flag:
    s = f.readline()
    if s:
#        print(i,s)
        ss = s.split(",")
        g.append(float(ss[1]))
        i = i + 1
    else:
        # reached end of file; stop while loop
        flag = False
        
print("# of measurements read is",i)
f.close()

#plt.subplot(110)


nbins = 100

plt.figure(figsize=(10,10))
counts, edges, something  = plt.hist(g,nbins, color = 'r')

g_max = np.amax(g)
g_ave = np.average(g)
g_std = np.std(g)
g_uncertainty = g_std/np.sqrt(i)

par = [g_max,g_ave,g_std]
g_max = np.amax(counts)
g_ave = np.average(g)
g_std = np.std(g)
g_uncertainty = g_std/np.sqrt(i)
print("Gaussian parameters:",par)
print("<g> = ",g_ave,"std(g) = ",g_std)
print("uncertainty sigma/sqrt(N) on <g> =",g_uncertainty)

plt.xlabel(r'g[m/$s^2$]', fontsize = 15)
plt.ylabel(r'#of measurements per bin', fontsize = 15)


- most of the data are close to 10m/s^2, but one data point is at 980 m/s^2
- data points like this are called outliers
- what to do with data like this?
- Do not just throw them away, but think how this might have happened
- is there an explanation? Was this a mistake or is this a big discovery?
- in this case, it is __likely__ that someone reported the result in cm/s^2 instead of m/s^2, making a factor of 100 difference
- let's do this again, but clean up the data before plotting

In [ ]:
#### import numpy as np
import matplotlib.pylab as plt
import numpy as np

# example of function definition
def gauss(x, a, x0, sigma):
   return a*np.exp(-(x-x0)**2/(2*(sigma**2)))

# open a text file containing data

f = open("../../Data/pendulumData.csv","r")

#read one line of data and discard

s = f.readline()

# define empty array to store data values
g = []


#the while loop will run until the end of file or i == nevent
flag = True
i = 0
while flag:
    s = f.readline()
    if s:
#        print(i,s)
        ss = s.split(",")
        x = float(ss[1])
        if(x > 500.0):
            x = x/100.0
        g.append(x)
        i = i + 1
    else:
        # reached end of file; stop while loop
        flag = False
        
print("# of measurements read is",i)
f.close()

nbins = 100

#range of histogram
xmin = 6
xmax = 16

plt.figure(figsize=(10,10))
counts, edges, something  = plt.hist(g,nbins, color = 'r',range = [xmin,xmax])


plt.xlabel(r'g[m/$s^2$]', fontsize = 15)
plt.ylabel(r'#of measurements per bin', fontsize = 15)


- Ok, that looks better
- now we can look at the important statistical properties of the distribution

In [ ]:
g_max = np.amax(g)
g_ave = np.average(g)
g_std = np.std(g)
g_uncertainty = g_std/np.sqrt(i)

print("<g> = ",g_ave,"std(g) = ",g_std)
print("uncertainty sigma/sqrt(N) on <g> =",g_uncertainty)


- we can do better by just plotting the __significant digits__
- out uncertainty is about 0.04. So digits much smaller than this are just noise with no information

In [ ]:
print("Final result:")
print(f'<g> = {g_ave:.3f}',f'std(g) = {g_std:.3f}')
print(f'uncertainty sigma/sqrt(N) on <g> = {g_uncertainty:.3f}')

In [ ]:
#Import the curve_fit function from scipy

from scipy.optimize import curve_fit

# Define a Gaussian function
def gauss(x, a, x0, sigma):
   return a*np.exp(-(x-x0)**2/(2*(sigma**2)))

# set starting values for the Gaussian fit
par = [g_max,g_ave,g_std]

g_uncertainty = g_std/np.sqrt(i)
print("Gaussian parameters:",par)

# calculate the uncertainty on each bin as sqrt(entries)
y_err = []
x_val = []
y_val = []
for i in range(len(counts)):
    e = np.sqrt(counts[i])
    if e > 0:
        y_err.append(e)
        x_val.append((edges[i]+edges[i+1])/2)
        y_val.append(counts[i])

plt.errorbar(x_val,y_val,yerr=y_err,fmt='ko')
plt.xlabel(r'g[m/$s^2$]', fontsize = 15)
plt.ylabel(r'#of measurements per bin', fontsize = 15)

#using the starting parameters, fit a Gaussian curve to the data
popt, pcov = curve_fit(gauss, x_val, y_val,p0 = par, sigma = y_err)

# get parameter errors as sqrt of diagonal of covariance matrix
perr = np.sqrt(np.diag(pcov))

# calculate the chi^2
chi2 = 0.0
for i in range(len(x_val)):
    x = (gauss(x_val[i],*popt) - y_val[i])/y_err[i]
    chi2 = chi2 + x*x
    
ndof = len(x_val) - len(par)

print("Chi^2 = ",chi2,"for ",ndof,"#DOF")
    
print("Fitted parameters = ",popt)
print("Uncertainties = ",perr)

# plot Gaussian with fitted parameters
xx = np.arange(xmin,xmax,0.1)
plt.plot(xx, gauss(xx, *popt),'g',label="Gaussian fit")

We see a chi^2 of 103.5 for 34 degrees of freedom. Is that good or bad? We can calculate the __chi^2 probability__

For this we need to understand two things:
- The chi^2 for a given number of degress of freedom, coming from measurements with normal-distributed uncertainties, itself has a statistical distribution. The distribution __only__ depends on the number of degrees of freedom
- If one knows the chi^2 and # DoF one can therefore calculate, using the __cumulative distribution__, how often one should expect to see a chi^2 that is equal or smaller than the one observed. E.g., if the chi^2 is very large, the corresponding chi^2-probability is very small and vice versa.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2

# -----------------------------
# Increase default font sizes
# -----------------------------
plt.rcParams.update({
    "font.size": 14,        # base font size
    "axes.titlesize": 16,   # title size
    "axes.labelsize": 14,   # axis labels
    "legend.fontsize": 12   # legend text
})

# -----------------------------
# Settings
# -----------------------------
degrees_of_freedom = 34
num_samples = 100_000
random_seed = 42

# -----------------------------
# Generate chi-square samples
# -----------------------------
rng = np.random.default_rng(random_seed)
samples = rng.chisquare(df=degrees_of_freedom, size=num_samples)

# -----------------------------
# Prepare x-values for theory curves
# -----------------------------
x_max = chi2.ppf(0.999, df=degrees_of_freedom)
x = np.linspace(0, x_max, 1000)

pdf = chi2.pdf(x, df=degrees_of_freedom)
cdf = chi2.cdf(x, df=degrees_of_freedom)

# -----------------------------
# Create the figure
# -----------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# -----------------------------
# Left plot: histogram + PDF
# -----------------------------
axes[0].hist(
    samples,
    bins=60,
    density=True,
    alpha=0.7,
    edgecolor="black",
    label="Simulated samples"
)

axes[0].plot(x, pdf, linewidth=2, label="Theoretical chi-square PDF")
axes[0].set_title(f"Chi-square Distribution (df = {degrees_of_freedom})")
axes[0].set_xlabel("Value")
axes[0].set_ylabel("Density")
axes[0].legend()

# -----------------------------
# Right plot: cumulative distribution
# -----------------------------
axes[1].plot(x, cdf, linewidth=2, label="Theoretical CDF")
axes[1].set_title(f"Cumulative Distribution (df = {degrees_of_freedom})")
axes[1].set_xlabel("Value")
axes[1].set_ylabel("Cumulative probability")
axes[1].set_ylim(0, 1.0)
axes[1].legend()

plt.tight_layout()
plt.show()

How often should we see a chi^2 as large as 103.5? From the cumulative distribution we see that the chi^2 should almost always be smaller than 103.5
We can calculate the exact value of the probability using numpy:

In [ ]:
import numpy as np
from scipy.stats import chi2

# ---------------------------------------
# This program:
# 1. Prompts the user for a chi-square value and degrees of freedom (DoF)
# 2. Computes the p-value (right-tail probability)
#    i.e., P(Chi-square >= observed value)
# ---------------------------------------

# Prompt user for inputs
chi2_value = float(input("Enter the chi-square value: "))
degrees_of_freedom = int(input("Enter the degrees of freedom: "))

# ---------------------------------------
# Calculate the p-value
# ---------------------------------------
# The survival function (sf) gives P(X >= x),
# which is the typical p-value for chi-square tests
p_value = chi2.sf(chi2_value, df=degrees_of_freedom)

# ---------------------------------------
# Also calculate the cumulative probability (CDF)
# This gives P(X <= x)
# ---------------------------------------
cdf_value = chi2.cdf(chi2_value, df=degrees_of_freedom)

# ---------------------------------------
# Display results
# ---------------------------------------
print("\nResults:")
print(f"Chi-square value: {chi2_value}")
print(f"Degrees of freedom: {degrees_of_freedom}")

# Right-tail probability (most commonly used p-value)
print(f"P(Chi-square >= {chi2_value}) = {p_value:.6f}")

# Left-tail cumulative probability
print(f"P(Chi-square <= {chi2_value}) = {cdf_value:.6f}")

The probability of seeing such a large a chi^2 is __0.00000__. What is wrong with our measurement? Let's look at the data again. 

In [ ]:
# Import the curve_fit function from scipy
from scipy.optimize import curve_fit
import numpy as np
import matplotlib.pyplot as plt

# -------------------------------------------------
# Define a double Gaussian with a shared mean
# -------------------------------------------------
def double_gauss_shared_mean(x, a1, a2, x0, sigma1, sigma2):
    return (
        a1 * np.exp(-(x - x0)**2 / (2 * sigma1**2)) +
        a2 * np.exp(-(x - x0)**2 / (2 * sigma2**2))
    )

# -------------------------------------------------
# Build x, y, and uncertainty arrays
# -------------------------------------------------
y_err = []
x_val = []
y_val = []

for i in range(len(counts)):
    e = np.sqrt(counts[i])
    if e > 0:
        y_err.append(e)
        x_val.append((edges[i] + edges[i + 1]) / 2)
        y_val.append(counts[i])

x_val = np.array(x_val)
y_val = np.array(y_val)
y_err = np.array(y_err)

# -------------------------------------------------
# Initial parameter guesses
# a1, a2: amplitudes
# x0: shared mean
# sigma1, sigma2: widths
# -------------------------------------------------
a_guess = np.max(y_val)
x0_guess = x_val[np.argmax(y_val)]

par = [
    a_guess / 2,      # a1
    a_guess / 2,      # a2
    x0_guess,         # shared mean
    g_std,            # sigma1
    2 * g_std         # sigma2 (make them different to help fit)
]

print("Starting parameters:", par)

# -------------------------------------------------
# Plot data
# -------------------------------------------------
plt.errorbar(x_val, y_val, yerr=y_err, fmt='ko')
plt.xlabel(r'g[m/$s^2$]', fontsize=15)
plt.ylabel(r'# of measurements per bin', fontsize=15)

# -------------------------------------------------
# Bounds (keep parameters physical)
# -------------------------------------------------
lower_bounds = [0, 0, xmin, 0, 0]
upper_bounds = [np.inf, np.inf, xmax, np.inf, np.inf]

# -------------------------------------------------
# Fit
# -------------------------------------------------
popt, pcov = curve_fit(
    double_gauss_shared_mean,
    x_val,
    y_val,
    p0=par,
    sigma=y_err,
    bounds=(lower_bounds, upper_bounds),
    maxfev=10000
)

# Parameter uncertainties
perr = np.sqrt(np.diag(pcov))

# -------------------------------------------------
# Chi-square calculation
# -------------------------------------------------
chi2_val = 0.0
for i in range(len(x_val)):
    resid = (double_gauss_shared_mean(x_val[i], *popt) - y_val[i]) / y_err[i]
    chi2_val += resid * resid

ndof = len(x_val) - len(par)

print("Chi^2 =", chi2_val, "for", ndof, "#DOF")
print("Fitted parameters =", popt)
print("Uncertainties =", perr)

# -------------------------------------------------
# Plot fit
# -------------------------------------------------
xx = np.arange(xmin, xmax, 0.1)
plt.plot(xx, double_gauss_shared_mean(xx, *popt), 'g', label="Double Gaussian (shared mean)")
plt.legend()
plt.show()

In [ ]:
p_value = chi2.sf(chi2_val, df=ndof)
print("Chi^2 probability for the double-gaussian fit:",p_value)

In [ ]:
plt.errorbar(x_val, y_val, yerr=y_err, fmt='ko')
plt.xlabel(r'g[m/$s^2$]', fontsize=15)
plt.ylabel(r'# of measurements per bin', fontsize=15)
plt.plot(xx, double_gauss_shared_mean(xx, *popt), 'g', label="Double Gaussian (shared mean)")
plt.yscale("log")
plt.ylim(0.5, 250)
plt.legend()
plt.show()